# Visual-token budget sweep — DeepSeek-OCR-2 on TexOCR-eval pages

Three native configurations, whole papers, resume-safe.

| run name | `base_size` | `image_size` | `crop_mode` | nominal visual tokens |
|---|---|---|---|---|
| `Base` | 1024 | 1024 | False | 256 |
| `Large` | 1280 | 1280 | False | 400 |
| `Gundam` | 1024 | 768 | True | 256 + n x 144, n = 0..6 |

Outputs (on Drive, so a timeout costs nothing):

- `runs.csv` — one row per (mode, page), appended and fsynced immediately; rerunning skips
  pairs already present, so a 4–6 h sweep can span several sessions.
- `outputs/<mode>/<page_id>.tex` — raw model output, one file per page.

Compilation is **not** done here: run `scripts/score.py --group-by paper` on a CPU runtime
afterwards (pdflatex on a GPU runtime burns units for nothing).

Prereqs: `data/pages.jsonl`, `data/images/*.png`, `data/paper_gold/*.tex` uploaded to Drive
(produced by `scripts/build_pages.py` on the laptop).

In [ ]:
# 1. Environment. Runs once per session (~4 min).
!pip install -q "transformers==4.46.3" "tokenizers<0.21" accelerate addict easydict rapidfuzz pandas matplotlib
%env TOKENIZERS_PARALLELISM=false
import torch, sys
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpu", torch.cuda.get_device_name(0))
print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# 2. Paths. Everything durable lives on Drive; code is a fresh clone.
from google.colab import drive
drive.mount("/content/drive")

import os, json, pathlib
WORK = pathlib.Path("/content/drive/MyDrive/ocr-layout-budget")   # <-- edit if different
REPO = pathlib.Path("/content/ocr-layout-token-budget")
DATA = WORK / "data"
OUT  = WORK / "outputs"
RUNS = WORK / "runs.csv"
for p in (OUT / "Base", OUT / "Large", OUT / "Gundam"):
    p.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    !git clone -q https://github.com/hluaguo/ocr-layout-token-budget {REPO}

pages = [json.loads(l) for l in open(DATA / "pages.jsonl")]
print(len(pages), "pages,", len({p['paper_id'] for p in pages}), "papers")
missing = [p['page_id'] for p in pages if not (DATA / p['image']).exists()]
print("missing images:", len(missing), missing[:3])

In [ ]:
# 3. Load the model (once per session, ~2 min).
import inspect, transformers
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "deepseek-ai/DeepSeek-OCR-2"
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID, trust_remote_code=True, use_safetensors=True,
    _attn_implementation="eager",   # flash_attention_2 if the wheel is present
).eval().cuda().to(torch.bfloat16)

print("infer signature:", inspect.signature(model.infer))
PROMPT = "<image>\n<|grounding|>Convert the document to latex."

### Config table + token accounting

`measured_visual_tokens` is computed from the config actually passed:

`(base_size/64)^2 + num_crops * (image_size/64)^2` → 256 / 400 / 256 + n*144

`num_crops` is taken from a forward hook on the vision tower when one can be found,
otherwise it is the documented maximum for the page's aspect ratio and `token_source`
says `formula_approx`. Never mix the two silently — the column is in `runs.csv`.

In [ ]:
CONFIGS = {
    "Base":   dict(base_size=1024, image_size=1024, crop_mode=False),
    "Large":  dict(base_size=1280, image_size=1280, crop_mode=False),
    "Gundam": dict(base_size=1024, image_size=768,  crop_mode=True),
}
TOK_PER_PATCH = 64           # patch side in px after the compressor: (size/64)^2 tokens

def nominal_tokens(cfg, num_crops):
    base = (cfg["base_size"] // TOK_PER_PATCH) ** 2
    if not cfg["crop_mode"]:
        return base, 0
    per_crop = (cfg["image_size"] // TOK_PER_PATCH) ** 2
    return base + num_crops * per_crop, num_crops

for name, cfg in CONFIGS.items():
    print(f"{name:7s} {cfg}  ->  base 256 + n*144  (max {nominal_tokens(cfg, 6)[0]} tokens)")

# best-effort hook: capture the number of crops the encoder actually consumes
VISION_ATTRS = ("vision_model", "vision_tower", "vision_encoder", "deepencoder", "encoder")
HOOK_STATE = {"n_crops": None}
def _hook_factory():
    def hook(module, args, kwargs, output):
        for obj in list(args) + list(kwargs.values()):
            if torch.is_tensor(obj) and obj.dim() == 5:      # (B, n, C, H, W)
                HOOK_STATE["n_crops"] = int(obj.shape[1]); return
            if isinstance(obj, (list, tuple)) and len(obj) and torch.is_tensor(obj[0]) and obj[0].dim() == 4:
                HOOK_STATE["n_crops"] = len(obj); return
    return hook

hooked = []
for attr in VISION_ATTRS:
    mod = getattr(model, attr, None)
    if mod is not None and hasattr(mod, "register_forward_hook"):
        mod.register_forward_hook(_hook_factory())
        hooked.append(attr)
print("hooks on:", hooked or "none (will fall back to formula_approx)")

In [ ]:
# 4. Resume-safe index of what already ran.
import csv, os, time
FIELDS = ["mode", "page_id", "paper_id", "page", "bucket", "base_size", "image_size",
          "crop_mode", "num_crops", "measured_visual_tokens", "token_source",
          "pred_path", "gold_paper", "gen_seconds", "pred_chars"]

def load_done(path):
    done = set()
    if not path.exists():
        return done
    with open(path, newline="", encoding="utf-8", errors="replace") as fh:
        for row in csv.DictReader(fh):
            if row.get("mode") and row.get("page_id"):
                done.add((row["mode"], row["page_id"]))
    return done

done = load_done(RUNS)
todo = [(m, p) for m in CONFIGS for p in pages if (m, p["page_id"]) not in done]
print("already done:", len(done), "| to run:", len(todo), "| total:", len(CONFIGS) * len(pages))

if not RUNS.exists():
    with open(RUNS, "w", newline="", encoding="utf-8") as fh:
        csv.DictWriter(fh, fieldnames=FIELDS).writeheader()
        fh.flush(); os.fsync(fh.fileno())

In [ ]:
# 5. The sweep. Each row is flushed + fsynced before the next page, so a Colab
#    timeout loses at most one generation. Re-run this cell to resume.
import time, traceback

RUN_LIMIT = 40        # pages per cell run; keeps the session under the 4-6 h budget
ran = 0

for mode, page in todo:
    if ran >= RUN_LIMIT:
        print("run limit reached -- re-run this cell to continue")
        break
    cfg = CONFIGS[mode]
    img = DATA / page["image"]
    dest = OUT / mode / (page["page_id"] + ".tex")
    t0 = time.time()
    HOOK_STATE["n_crops"] = None
    try:
        text = model.infer(
            tok,
            prompt=PROMPT,
            image_file=str(img),
            output_path=None,
            base_size=cfg["base_size"],
            image_size=cfg["image_size"],
            crop_mode=cfg["crop_mode"],
            save_results=False,
        )
        if isinstance(text, (list, tuple)):
            text = text[0] if text else ""
        text = text or ""
    except Exception:
        print("FAIL", mode, page["page_id"])
        traceback.print_exc(limit=3)
        continue

    dest.write_text(text, encoding="utf-8")
    n_crops = HOOK_STATE["n_crops"]
    source = "hook"
    if n_crops is None:
        # documented maximum for the configured mode; the aspect ratio decides
        # how many are used, so this is an upper bound, labelled as such
        n_crops = 6 if cfg["crop_mode"] else 0
        source = "formula_approx"
    tokens, n_crops = nominal_tokens(cfg, n_crops)

    with open(RUNS, "a", newline="", encoding="utf-8") as fh:
        csv.DictWriter(fh, fieldnames=FIELDS).writerow({
            "mode": mode, "page_id": page["page_id"], "paper_id": page["paper_id"],
            "page": page["page"], "bucket": page["bucket"],
            "base_size": cfg["base_size"], "image_size": cfg["image_size"],
            "crop_mode": cfg["crop_mode"], "num_crops": n_crops,
            "measured_visual_tokens": tokens, "token_source": source,
            "pred_path": str(dest), "gold_paper": str(DATA / page["paper_gold"]),
            "gen_seconds": round(time.time() - t0, 2), "pred_chars": len(text),
        })
        fh.flush(); os.fsync(fh.fileno())
    ran += 1
    print(f"{ran:3d} {mode:7s} {page['page_id']} {tokens:5d}tok {time.time()-t0:5.1f}s {len(text):6d}ch")

print("this cell:", ran, "| total logged:", len(load_done(RUNS)))

In [ ]:
# 6. Deterministic output processing + quick sanity table, then scoring.
#    TexOCR's harness strips code fences; do the same here so NED is comparable.
def process_rsp(r):
    if not r:
        return ""
    for fence in ("```latex", "```bibtex", "```"):
        if r.startswith(fence):
            return r[len(fence):].rstrip("`").rstrip()
    return r

for f in sorted(OUT.rglob("*.tex")):
    raw = f.read_text(encoding="utf-8", errors="replace")
    clean = process_rsp(raw)
    if clean != raw:
        f.write_text(clean, encoding="utf-8")

import collections, pandas as pd
df = pd.read_csv(RUNS)
print(df.groupby(["bucket", "mode"])[["measured_visual_tokens", "gen_seconds", "pred_chars"]]
        .agg(["count", "mean"]).round(1))
print("\nrows with zero output:", int((df.pred_chars == 0).sum()))

In [ ]:
# 7. Score the completed subset (CPU pass; no pdflatex here).
#    --group-by paper concatenates each paper's pages against paper_gold, exactly
#    like TexOCR's harness, so NED is comparable to their edit_distance column.
import subprocess
cmd = ["python", "scripts/score.py", "--runs", str(RUNS), "--pages", str(DATA / "pages.jsonl"),
       "--group-by", "paper", "--no-compile", "--out", str(WORK / "scores.csv"), "--jobs", "8"]
res = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
print(res.stderr[-2500:])

import pandas as pd
display(pd.read_csv(WORK / "scores_summary.csv"))
display(pd.read_csv(WORK / "scores_paper_summary.csv"))

In [ ]:
# 8. The plot: measured visual tokens on x, text + layout metrics on y.
#    Layout = compile pass rate once the CPU pdflatex pass is done; until then only
#    tabular parity is available, so the figure is written with whatever exists.
import matplotlib.pyplot as plt, pandas as pd

paper = pd.read_csv(WORK / "scores_paper.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
order = ["Base", "Large", "Gundam"]

g = paper.groupby("mode").agg(tokens=("vision_tokens_mean", "mean"),
                               ned=("ned", "mean"))
ax[0].plot(g.tokens, 1 - g.ned, "o-", label="1 - NED (text)")
ax[0].set_ylabel("text fidelity"); ax[0].set_title("Text survives")

if paper["compile_ok"].notna().any():
    c = paper.groupby("mode").agg(tokens=("vision_tokens_mean", "mean"),
                                   rate=("compile_ok", "mean"))
    ax[1].plot(c.reindex([m for m in order if m in c.index]).tokens,
               c.reindex([m for m in order if m in c.index]).rate, "s-", label="pdflatex pass rate")
    ax[1].set_title("Structure does not")
else:
    ax[1].text(0.5, 0.5, "run the CPU pdflatex pass", ha="center", va="center")
ax[1].set_ylabel("compile pass rate")
for a in ax:
    a.set_xlabel("measured visual tokens / page"); a.grid(alpha=.3); a.legend()
fig.tight_layout(); fig.savefig(WORK / "figs" / "budget_vs_layout.png", dpi=200)
print("wrote", WORK / "figs" / "budget_vs_layout.png")